# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AmaanJilani1/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
import duckdb, numpy as np

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
else:
    import os
    HF_TOKEN = os.environ.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"
FACT_MONTH = f"{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet"
CONTENT = f"{BASE}/dim_content.parquet"

print("dim_content columns (for join/feature candidates):")
print(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{CONTENT}')").df())


dim_content columns (for join/feature candidates):
                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  No

## 1. Build the feature vector

Builds a wider content-level feature frame for `month=2026-03`: engineered ratios, categorical handling for content attributes from `dim_content` (if present), and explicit fills for missing values (rather than silently letting `NULL`s propagate). Rows are filtered to `gsc_data_available IS TRUE`, since a row without GSC data can't honestly contribute to GSC-based aggregates.

In [2]:
raw = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        AVG(gsc_avg_position) AS mean_position,
        STDDEV(gsc_avg_position) AS position_volatility,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS ctr,
        COUNT(DISTINCT report_date) AS days_with_data,
        SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END)
          - SUM(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_impressions ELSE 0 END)
          AS impressions_trend_delta
    FROM read_parquet('{FACT_MONTH}')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

# Join content-level attributes from dim_content on content_hash_id.
content_dim = con.sql(f"SELECT * FROM read_parquet('{CONTENT}')").df()
features = raw.merge(content_dim, on="content_hash_id", how="left")

# Explicit fills, not silent NaNs:
# - ratios where the denominator was 0 (no impressions all month) -> fill ctr with 0, not NaN
features["ctr"] = features["ctr"].fillna(0)
# - volatility is undefined (NaN) when a page only had 1 day of data -> fill with 0 (no observed volatility)
features["position_volatility"] = features["position_volatility"].fillna(0)

print(f"Feature frame shape: {features.shape}")
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (176738, 34)


,client_hash_id_x,content_hash_id,total_impressions,total_clicks,mean_position,position_volatility,ctr,days_with_data,impressions_trend_delta,client_hash_id_y,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,7.209549,2.442255,0.001073,31,-1823.0,client_73cda7b4e4f265ea,...,0,2025-02-28,None,gemini-3-flash-preview,13886,2123,2026-07-06,2026-08-20,True,False
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,2.987198,2.481312,0.000000,31,-37.0,client_73cda7b4e4f265ea,...,0,2025-02-28,None,None,<NA>,<NA>,NaT,NaT,True,False
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,6.724039,1.243547,0.001066,31,-1780.0,client_73cda7b4e4f265ea,...,0,2025-02-28,None,gemini-3-flash-preview,16204,2546,2026-05-20,2026-07-04,True,False
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,7.244844,0.983020,0.002629,31,64.0,client_73cda7b4e4f265ea,...,0,2025-02-28,None,gemini-3-flash-preview,15430,2330,2026-07-06,2026-08-20,True,False
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,4.209227,4.341012,0.002331,31,-51.0,client_73cda7b4e4f265ea,...,0,2025-02-28,None,None,<NA>,<NA>,NaT,NaT,True,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature | Meaning | Missing handling | Available before decision moment? |
|---|---|---|---|
| `total_impressions` | sum of daily `gsc_impressions` across March | none expected (SUM of filtered, available rows) | Yes — purely historical, aggregated only up to month end |
| `total_clicks` | sum of daily `gsc_clicks` across March | same as above | Yes — historical only |
| `mean_position` | average daily `gsc_avg_position` across the month | NaN only if a page has zero surviving rows for the month (excluded by the GROUP BY/filter itself) | Yes — historical only |
| `position_volatility` | standard deviation of daily position (how much rank swings day to day) | NaN when only 1 day of data exists (STDDEV undefined for n=1) -> explicitly filled with 0 | Yes — historical only |
| `ctr` | total_clicks / total_impressions | NaN when impressions = 0 for the month -> explicitly filled with 0, not silently dropped | Yes — historical only |
| `days_with_data` | count of distinct `report_date` rows in March with GSC data available for that page | none expected | Yes — a diagnostic/context feature, not really predictive, but useful to flag thin-data pages |
| content attributes joined from `dim_content` | static or slow-changing page attributes (exact fields per the Section 0 printout) | any NULLs from the left join (content present in fact table but missing from dim_content) treated as "unknown", not silently zero-filled, since 0 could be misread as a real numeric/category value | Yes — content attributes precede any given day's performance by construction |
| `impressions_trend_delta` | second-half minus first-half March `gsc_impressions` | 0 possible if page had identical impressions in both halves (not missing, a real value) | **This is the label proxy, not a feature — kept separate on purpose (see Section 3).** |

**Categorical handling note:** any text/categorical column from `dim_content` needs one-hot or target-safe encoding before modeling — not left as raw strings. Confirm exact fields via the Section 0 printout before encoding, since guessing field names caused real errors earlier in this notebook's development.

In [3]:
# Quick audit: confirm the fills above actually resolved the NaNs they claim to
print("Remaining NaN counts per column after fills:")
print(features[["total_impressions", "total_clicks", "mean_position",
                 "position_volatility", "ctr", "days_with_data"]].isna().sum())

Remaining NaN counts per column after fills:
total_impressions      0
total_clicks           0
mean_position          0
position_volatility    0
ctr                    0
days_with_data         0
dtype: int64


## 3. The leakage hunt

Going wider than the single deliberate-trap column from `w03_data_contract.ipynb`: here I attack **every** candidate feature and ask, for each, whether it could only be known *because* the outcome already happened.

In [4]:
# Define the proxy label the same way as w03_data_contract.ipynb, kept OUT of the feature set.
features["is_declining"] = (features["impressions_trend_delta"] < 0).astype(int)

candidate_features = ["total_impressions", "total_clicks", "mean_position",
                       "position_volatility", "ctr", "days_with_data"]

# Leakage test 1: correlation of each candidate feature with the label-derived quantity itself.
print("Correlation of each candidate feature with impressions_trend_delta (the value the label is derived from):")
for f in candidate_features:
    corr = features[f].corr(features["impressions_trend_delta"])
    flag = "  <-- SUSPECT, investigate" if abs(corr) > 0.6 else ""
    print(f"  {f}: {corr:.3f}{flag}")


Correlation of each candidate feature with impressions_trend_delta (the value the label is derived from):
  total_impressions: 0.399
  total_clicks: 0.460
  mean_position: -0.057
  position_volatility: -0.048
  ctr: 0.000
  days_with_data: 0.038


In [5]:
# Leakage test 2: leave-one-feature-out, to see if any single "honest" feature is doing
# suspiciously all the work (could indicate a hidden restatement of the label).
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

X_all = features[candidate_features].fillna(0)
y = features["is_declining"]
X_train, X_test, y_train, y_test = train_test_split(X_all, y, test_size=0.2, random_state=42)

full_tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X_train, y_train)
full_score = full_tree.score(X_test, y_test)
print(f"Full honest feature set accuracy: {full_score:.3f}\n")

for f in candidate_features:
    remaining = [c for c in candidate_features if c != f]
    t = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
    t.fit(X_train[remaining], y_train)
    s = t.score(X_test[remaining], y_test)
    drop = full_score - s
    print(f"Without {f:22s} accuracy: {s:.3f}  (drop of {drop:+.3f})")


Full honest feature set accuracy: 0.539

Without total_impressions      accuracy: 0.614  (drop of -0.075)
Without total_clicks           accuracy: 0.526  (drop of +0.013)
Without mean_position          accuracy: 0.550  (drop of -0.011)
Without position_volatility    accuracy: 0.539  (drop of +0.000)
Without ctr                    accuracy: 0.539  (drop of +0.000)
Without days_with_data         accuracy: 0.496  (drop of +0.043)


In [6]:
# Leakage test 3 (the deliberate trap, repeated here for completeness): add the literal
# label-derived column back in and confirm the score jumps toward-perfect, then remove it.
X_leaky = features[candidate_features + ["impressions_trend_delta"]].fillna(0)
Xl_train, Xl_test, _, _ = train_test_split(X_leaky, y, test_size=0.2, random_state=42)
leaky_tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(Xl_train, y_train)
leaky_score = leaky_tree.score(Xl_test, y_test)
print(f"HONEST (no leak) accuracy: {full_score:.3f}")
print(f"LEAKY accuracy (impressions_trend_delta included): {leaky_score:.3f}  <- jumps toward 1.0")
print("Confirms the trap: this column is the label's source, not an independent signal.")

del X_leaky
print(f"\nKeeping the honest number: {full_score:.3f}")


HONEST (no leak) accuracy: 0.539
LEAKY accuracy (impressions_trend_delta included): 1.000  <- jumps toward 1.0
Confirms the trap: this column is the label's source, not an independent signal.

Keeping the honest number: 0.539


## 4. What I excluded and why

| Excluded field | Why |
|---|---|
| `impressions_trend_delta` | this is the exact quantity the label (`is_declining`) is derived from — including it as a feature is leakage by definition (confirmed in Section 3, test 3) |
| `fact_content_query_90d.parquet` (whole table) | different grain (content x query, not content x day) — mixing grains changes the unit of analysis without re-deriving the contract |
| `ga4_*` / `sessions_*` / `ai_*` fields | GA4 on-site behavior and traffic-source signals — out of scope for this search-visibility lane; mixing them in changes the question being asked |
| `client_hash_id` as a model feature (kept only as a context/grouping key) | using it as a predictive feature would let the model key off *which client* a page belongs to rather than the page's own signals — also needed intact, unencoded, for the client-holdout split in later weeks |
| raw text/identifying fields from `dim_content` (if any, e.g. URLs or titles) | high re-identification risk and not meaningfully predictive in raw form; no client-identifying detail belongs in any feature set or public output |
| final month (June 2026 / `fact_content_daily_performance_sample.parquet`) | sealed test month — using it now, even just to "peek," would leak future information into decisions made about March |

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.